## Search Engine With Tools And Agents

In [5]:
## Arxiv - Search (Research Paper Search Engine)
## Tools Creation

from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper,ArxivAPIWrapper

C:\Users\baibh\AppData\Local\Temp\ipykernel_11760\173614880.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import ArxivQueryRun,WikipediaQueryRun


In [6]:
## Used the inbuilt tool of wikipedia

api_wrapper_wiki = WikipediaAPIWrapper(top_k_results = 3, doc_content_char_max=250)
wiki = WikipediaQueryRun(api_wrapper = api_wrapper_wiki)
wiki.name

'wikipedia'

In [7]:
api_wrapper_arxiv = ArxivAPIWrapper(top_k_results=1, doc_content_chars_max=250)
arxiv = ArxivQueryRun(api_wrapper = api_wrapper_arxiv)
arxiv.name

'arxiv'

In [8]:
tools = [wiki,arxiv]

In [9]:
## Custom tools

from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [10]:
loader = WebBaseLoader("https://docs.smith.langchain.com/")
docs = loader.load()
docs = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_documents(docs)
vectordb = FAISS.from_documents(docs,HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
retriever = vectordb.as_retriever()
retriever

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1680.23it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B008845BB0>, search_kwargs={})

In [11]:
from langchain_classic.tools.retriever import create_retriever_tool
retriever_tool = create_retriever_tool(retriever,'langsmith_search','Search any information related to LangSmith')

retriever_tool.name

'langsmith_search'

In [12]:
tools = [wiki,arxiv,retriever_tool]
tools

[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'C:\\Users\\baibh\\AppData\\Local\\Packages\\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\\LocalCache\\local-packages\\Python312\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)),
 ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=250)),
 StructuredTool(name='langsmith_search', description='Search any information related to LangSmith', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x000001B008C605E0>, coroutine=<function create_retriever_tool.<locals>.afun

In [13]:
## Run all these tools with Agents and LLM Models

## Tools, LLMs --> Agent Executor
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')

llm = ChatGroq(groq_api_key = groq_api_key,model_name = "llama-3.3-70b-versatile")

In [14]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_prompt = """You are a helpful AI assistant.

You have access to the following tools:
- Wikipedia Search
- Arxiv Search
- LangSmith Retriever

Use the appropriate tool whenever external information is required.
If the answer is not available from your tools, answer using your general knowledge.
Always provide clear and concise responses."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        MessagesPlaceholder(variable_name="chat_history", optional=True),
        ("human", "{input}"),
        MessagesPlaceholder(variable_name="agent_scratchpad"),
    ]
)

In [15]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
    You are a helpful AI assistant.
    Use the available tools whenever needed.
    """
)

In [17]:
response = agent.invoke(
    {
        'messages' : [
            {'role':'user','content':'What is LangSmith?'}
        ]
    }
)

content = response['messages'][-1].content

print(content)

LangSmith is an observability platform that provides full visibility into LLM (Large Language Model) applications, from individual traces to production-wide performance metrics. It works with many frameworks and providers, including OpenAI, Anthropic, and Vercel AI SDK, and offers features such as tracing, monitoring, and automations to help developers build and optimize their LLM applications.
